# Milestone 5 lab, notebook 4 of 7: When a task fails

## Where we are

| Notebook | What you do |
|---|---|
| 1. A clock is not enough | Start two steps by the clock, and get yesterday's report |
| 2. Tasks and arrows | Read the DAG, and run it once |
| 3. Which day is this run for? | The logical date, the data interval, and a run you trigger by hand |
| **4. When a task fails** (you are here) | Break the API, then break the data, and watch what Airflow does |
| 5. The days you missed | Run a week with one command. Run one day again |
| 6. Every five minutes | A lab: the same pipeline as a micro-batch |
| 7. The bad morning | The drill: the API is down, then a page comes twice |

So far every run went well. A scheduler earns its place on the mornings that do not. This notebook breaks two different things, and Airflow must do two different things about them.

In [1]:
import time

from lab_helpers import at, MILESTONE_DIR, api, report_is_right, schedule_warehouse, show_doc, show_runs, start_fresh, table
from psycopg.types.json import Jsonb

at.wait_until_ready()
start_fresh()
at.trigger("2026-06-07")
at.trigger("2026-06-08")
show_runs()

Airflow has no runs. The warehouse is empty. The order system is back at its first morning.


,business_day,started_by,state
0,2026-06-07,manual,success
1,2026-06-08,manual,success


Two good mornings are loaded.

## Step 1: The order system is down

It is 07:00 on Wednesday. The order system answers every call with an error. Plan seven failures.

In [2]:
api.post("/admin/faults", json={"kind": "error_500", "count": 7})

at.trigger("2026-06-09", wait=False)        # start the run, and do not wait

seen = []
while True:
    state = at.task_states("2026-06-09").get("extract_and_load")
    if state and (not seen or seen[-1][1] != state):
        seen.append((time.strftime("%H:%M:%S"), state))
        print(f"{seen[-1][0]}   extract_and_load is {state}")
    if all(run["state"] in ("success", "failed") for run in at.runs()):
        break
    time.sleep(2)

16:10:56   extract_and_load is running


16:11:13   extract_and_load is up_for_retry


16:11:29   extract_and_load is running


16:11:35   extract_and_load is success


Follow the states.

1. `running`: the task starts. Inside it, the code of milestone 2 tries a call five times, waiting 1, 2, 4 and 8 seconds. All five fail.
2. `up_for_retry`: the task has failed, and **it has tries left**. Airflow waits 20 seconds.
3. `running`: Airflow starts the same task again. Two more calls fail, and then the order system answers.
4. `success`.

Nobody did anything. Look at the whole run.

In [3]:
print(at.task_states("2026-06-09"))
print("\nThe report equals the answer key for the morning of 10 June:", report_is_right("2026-06-10"))

{'set_source_clock': 'success', 'transform_and_test': 'success', 'publish_report': 'success', 'extract_and_load': 'success'}

The report equals the answer key for the morning of 10 June: True


The other tasks simply waited for `extract_and_load`, because of the arrows.

Trying a failed task again is a **retry**. It is set on the task:

In [4]:
dag_file = (MILESTONE_DIR / "dags" / "daily_orders_dag.py").read_text()
print(dag_file[dag_file.index("    # Read what changed and land it raw."):dag_file.index("    # Build the dbt models")])

    # Read what changed and land it raw. The API can be down, so this task may be tried again.
    extract_and_load = BashOperator(
        task_id="extract_and_load",
        bash_command=f"{PIPELINE} extract-load --run-date {RUN_DATE}",
        retries=3,
        retry_delay=timedelta(seconds=20),
    )




There are now two layers of retry, and they are for different sizes of trouble.

| Layer | Where | It handles |
|---|---|---|
| In the code (milestone 2) | Inside one API call: 5 tries in about 15 seconds | A hiccup of a few seconds |
| In Airflow (now) | The whole task again, after 20 seconds, up to 3 more times | An outage of a minute or two |

A retry of a whole task is safe for one reason only: the task is idempotent. It may have landed half its rows before it failed. Running it again must do no harm. That was milestone 3.

## Step 2: The data is wrong

Now a different kind of failure. The order system is up, and it sends a status nobody has seen. Land that bad row, and run Thursday.

In [5]:
con = schedule_warehouse()
payload = con.execute("SELECT payload FROM raw.orders_landing WHERE (payload->>'order_id')::bigint = 1130 ORDER BY landing_id DESC LIMIT 1").fetchone()[0]
payload.update(status="teleported", updated_at="2026-06-30T00:00:00Z")
con.execute("INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at) VALUES (%s, 'bad_data', 1, now())", [Jsonb(payload)])

at.trigger("2026-06-10")
show_runs()

,business_day,started_by,state
0,2026-06-07,manual,success
1,2026-06-08,manual,success
2,2026-06-09,manual,success
3,2026-06-10,manual,failed


In [6]:
at.task_states("2026-06-10")

{'transform_and_test': 'failed',
 'set_source_clock': 'success',
 'extract_and_load': 'success',
 'publish_report': 'upstream_failed'}

Read the four states.

- `extract_and_load`: `success`. The load was fine.
- `transform_and_test`: `failed`. A table test found the bad row, as in milestone 4.
- `publish_report`: `upstream_failed`. **It never ran.** A task before it failed.

And there was no retry this time. Look at how that task is written:

In [7]:
print(dag_file[dag_file.index("    # Build the dbt models"):dag_file.index("    publish_report = ")])

    # Build the dbt models and test the tables. No retries: a test that failed on the data
    # will fail again. A person must look.
    transform_and_test = BashOperator(
        task_id="transform_and_test",
        bash_command=f"{PIPELINE} transform-test",
    )




No `retries`. That is a decision.

> Do not retry a check that failed on the data.

The order system being down is a problem that **time** can fix. A row with a bad status is a problem only a **person** can fix. Running the test again in 20 seconds would find the same row, three times, and delay the moment someone looks.

The manager's report is still the last good one:

In [8]:
print("The report still equals the answer key for the morning of 10 June:", report_is_right("2026-06-10"))

The report still equals the answer key for the morning of 10 June: True


## Step 3: Someone fixes it, and runs the day again

The source is fixed: remove the bad row. Then run the failed day again. In Airflow that is called **clearing** the run: you clear its tasks, and Airflow runs them again.

In [9]:
con.execute("DELETE FROM raw.orders_landing WHERE run_id = 'bad_data'")
con.close()

at.replay("2026-06-10")
print(at.task_states("2026-06-10"))
print("\nThe report equals the answer key for the morning of 11 June:", report_is_right("2026-06-11"))

{'transform_and_test': 'success', 'set_source_clock': 'success', 'extract_and_load': 'success', 'publish_report': 'success'}

The report equals the answer key for the morning of 11 June: True


In [10]:
show_doc("HLD.md", "6. What happens when a task fails")

**From `docs/HLD.md`**

> ## 6. What happens when a task fails
> 
> *(There is a diagram here. Open `docs/HLD.md` to see it.)*
> 
> | Task | Retries | Why |
> |---|---|---|
> | `extract_and_load` | 3, with 20 seconds between | A source that is down often comes back |
> | `transform_and_test` | 0 | A test that failed on the data will fail again. Do not retry a check that failed on the data |
> | `publish_report` | 0 | It only reads |
> 
> A retry is only safe because every step is idempotent. That was milestone 3's work, and this milestone depends on it.

## A new word, and five states

| Word | Meaning | Here |
|---|---|---|
| **Retry** (of a task) | "A retry runs the failed task again after a short wait." | `extract_and_load`: up to 3 more times, 20 seconds apart |

| State | Meaning |
|---|---|
| `running` | The task is working now |
| `success` | It finished |
| `up_for_retry` | It failed, and it has tries left |
| `failed` | It failed, and has no tries left |
| `upstream_failed` | A task before it failed, so it never ran |

The definition is from the D8Loop chapter [Orchestration & Scheduling](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/orchestration-and-scheduling).

## What you now know

- When the order system was down, Airflow retried the load by itself, and the run succeeded.
- When the data was wrong, the test task failed once, and the report task never ran.
- Retry what time can fix. Do not retry what only a person can fix.
- A retry is safe only because the task is idempotent.

**Say it in an interview:** "I set retries per task, not per DAG. The extract task retries three times with a delay, because an API that is down usually comes back. The dbt test task has zero retries: if a test failed on the data it will fail again, so the run should stop and the downstream publish task goes to upstream_failed. And I can only retry a task at all because each one is idempotent."

Next: [notebook 5](05_the_days_you_missed.ipynb) runs a whole week with one command.